# 02 — The Three Kings

This notebook analyzes how Reddit talks about the three legends of Indian cricket - MS Dhoni, Virat Kohli, and Rohit Sharma - across cricket subreddits between June 2023 and July 2024.

### 1. Setup

Import utilities and start Spark.

In [ ]:
import sys
sys.path.insert(0, "..")

from utils import (
    get_spark, load_cricket_submissions, load_cricket_comments,
    add_player_mentions, add_time_features, save_figure, save_pandas_to_local,
    EVENT_DATES,
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, avg, sum as ssum, when, lower
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd

# Plot style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120


### 2. Start Spark

In [ ]:
# Start Spark
spark = get_spark("02_ThreeKings")
spark


### 3. Load filtered submissions

Load the cricket-filtered submissions saved in notebook 01.

In [ ]:
# Load filtered cricket submissions
subs = load_cricket_submissions(spark)
print(f"Submissions loaded")
subs.printSchema()


### 4. Load filtered comments

Load the cricket-filtered comments.

In [ ]:
# Load filtered cricket comments
coms = load_cricket_comments(spark)
print(f"Comments loaded")


### 5. Add time features

Add hour, day, month, year columns from the Unix timestamp.

In [ ]:
# Add time columns to submissions
subs = add_time_features(subs, ts_col="created_utc")

# Add time columns to comments
coms = add_time_features(coms, ts_col="created_utc")


### 6. Add player mention flags — submissions

Tag each submission with whether it mentions Dhoni, Kohli, or Rohit.

In [ ]:
# Tag player mentions in submission titles
subs = add_player_mentions(subs, text_col="title")


### 7. Add player mention flags — comments

Same for comments using the body text.

In [ ]:
# Tag player mentions in comment body
coms = add_player_mentions(coms, text_col="body")


### 8. Overall player mention counts — submissions

How many posts mention each player?

In [ ]:
# Count posts mentioning each player
mention_counts = subs.agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).toPandas()

print("Player mentions in submissions:")
print(mention_counts)


### 9. Overall player mention counts — comments

How many comments mention each player?

In [ ]:
# Count comments mentioning each player
com_mention_counts = coms.agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).toPandas()

print("Player mentions in comments:")
print(com_mention_counts)


### 10. Bar chart — player mentions in posts

In [ ]:
# Bar chart of player mentions
fig, ax = plt.subplots(figsize=(8, 5))
players = ["Dhoni", "Kohli", "Rohit"]
counts = [mention_counts["Dhoni"][0], mention_counts["Kohli"][0], mention_counts["Rohit"][0]]
colors = ["#f4a261", "#2a9d8f", "#e76f51"]

bars = ax.bar(players, counts, color=colors, width=0.5, edgecolor="white")
ax.bar_label(bars, labels=[f"{c:,}" for c in counts], padding=5, fontsize=11)
ax.set_title("Reddit Post Mentions: The Three Kings (Jun 2023 – Jul 2024)", fontsize=13)
ax.set_ylabel("Number of Posts")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
save_figure(fig, "02_player_mentions_posts.png")
plt.show()


### 11. Bar chart — player mentions in comments

In [ ]:
# Bar chart of player mentions in comments
fig, ax = plt.subplots(figsize=(8, 5))
com_counts = [com_mention_counts["Dhoni"][0], com_mention_counts["Kohli"][0], com_mention_counts["Rohit"][0]]

bars = ax.bar(players, com_counts, color=colors, width=0.5, edgecolor="white")
ax.bar_label(bars, labels=[f"{c:,}" for c in com_counts], padding=5, fontsize=11)
ax.set_title("Reddit Comment Mentions: The Three Kings (Jun 2023 – Jul 2024)", fontsize=13)
ax.set_ylabel("Number of Comments")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
save_figure(fig, "02_player_mentions_comments.png")
plt.show()


### 12. Monthly mention trends

How did player mentions change month by month?

In [ ]:
# Monthly mention counts per player
monthly = subs.filter(col("mentions_any_king")).groupBy("year_num", "month_num").agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).orderBy("year_num", "month_num").toPandas()

# Create a readable date label
monthly["date"] = pd.to_datetime(monthly[["year_num", "month_num"]].rename(
    columns={"year_num": "year", "month_num": "month"}
).assign(day=1))

print(monthly[["date", "Dhoni", "Kohli", "Rohit"]])


### 13. Line chart — monthly mention trends

In [ ]:
# Line chart of monthly mentions
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(monthly["date"], monthly["Dhoni"], marker="o", label="Dhoni", color="#f4a261", linewidth=2)
ax.plot(monthly["date"], monthly["Kohli"], marker="s", label="Kohli", color="#2a9d8f", linewidth=2)
ax.plot(monthly["date"], monthly["Rohit"], marker="^", label="Rohit", color="#e76f51", linewidth=2)

# Mark key events
ax.axvline(EVENT_DATES["wc2023_final"], color="red", linestyle="--", alpha=0.7, label="2023 WC Final")
ax.axvline(EVENT_DATES["t20wc2024_final"], color="gold", linestyle="--", alpha=0.7, label="T20 WC 2024 Final")

ax.set_title("Monthly Reddit Mentions: The Three Kings", fontsize=13)
ax.set_ylabel("Post Mentions")
ax.set_xlabel("Month")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
save_figure(fig, "02_monthly_mention_trends.png")
plt.show()


### 14. Engagement by player mention

Which player's posts get the highest average score and comment count?

In [ ]:
# Average score and comment count for posts mentioning each player
def player_engagement(df, player_col, player_name):
    return df.filter(col(player_col)).agg(
        count("*").alias("posts"),
        avg("score").alias("avg_score"),
        avg("num_comments").alias("avg_comments"),
    ).withColumn("player", F.lit(player_name))

dhoni_eng  = player_engagement(subs, "mentions_dhoni", "Dhoni")
kohli_eng  = player_engagement(subs, "mentions_kohli", "Kohli")
rohit_eng  = player_engagement(subs, "mentions_rohit", "Rohit")

engagement = dhoni_eng.union(kohli_eng).union(rohit_eng).toPandas()
print(engagement)
save_pandas_to_local(engagement, "02_player_engagement.csv")


### 15. Bar chart — average score by player

In [ ]:
# Bar chart of average post score per player
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Avg score
axes[0].bar(engagement["player"], engagement["avg_score"], color=colors, width=0.5, edgecolor="white")
axes[0].set_title("Avg Post Score by Player Mention")
axes[0].set_ylabel("Average Score")

# Avg comments
axes[1].bar(engagement["player"], engagement["avg_comments"], color=colors, width=0.5, edgecolor="white")
axes[1].set_title("Avg Comment Count by Player Mention")
axes[1].set_ylabel("Average Comments per Post")

plt.suptitle("Engagement: Which King Drives the Most Reddit Activity?", fontsize=13)
plt.tight_layout()
save_figure(fig, "02_player_engagement_charts.png")
plt.show()


### 16. Top subreddits per player

Where does each player get talked about most?

In [ ]:
# Top subreddits per player
def top_subs_for_player(df, player_col, player_name, n=10):
    return (
        df.filter(col(player_col))
        .groupBy("subreddit")
        .agg(count("*").alias("mentions"))
        .orderBy(col("mentions").desc())
        .limit(n)
        .toPandas()
        .assign(player=player_name)
    )

dhoni_subs = top_subs_for_player(subs, "mentions_dhoni", "Dhoni")
kohli_subs = top_subs_for_player(subs, "mentions_kohli", "Kohli")
rohit_subs = top_subs_for_player(subs, "mentions_rohit", "Rohit")

print("=== Dhoni ==="); print(dhoni_subs)
print("=== Kohli ==="); print(kohli_subs)
print("=== Rohit ==="); print(rohit_subs)


### 17. Stop Spark

In [ ]:
# Stop Spark
spark.stop()
